# IT-SUPPORT Data Acquisition

This notebook is the start of the project pipeline. Its job is to inspect candidate sources, track licensing, and run small acquisition slices. Downstream modelling decisions wait until we know what the data actually looks like.

In [1]:
from pathlib import Path
import json
import sys
!{sys.executable} -m pip install "ipywidgets>=8.1,<8.2"

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT / 'src'))
DATA_DIR = PROJECT_ROOT / 'data'
RAW_DIR = DATA_DIR / 'raw'
PROCESSED_DIR = DATA_DIR / 'processed'
ATTRIBUTION_DIR = DATA_DIR / 'attribution'

PROJECT_ROOT

  Using cached ipywidgets-8.1.8-py3-none-any.whl.metadata (2.4 kB)
  Using cached widgetsnbextension-4.0.15-py3-none-any.whl.metadata (1.6 kB)
  Using cached jupyterlab_widgets-3.0.16-py3-none-any.whl.metadata (20 kB)
Using cached ipywidgets-8.1.8-py3-none-any.whl (139 kB)
Using cached jupyterlab_widgets-3.0.16-py3-none-any.whl (914 kB)
Using cached widgetsnbextension-4.0.15-py3-none-any.whl (2.2 MB)



[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


WindowsPath('c:/Users/nwagb/Desktop/it_support_ai')

In [2]:
registry_path = DATA_DIR / 'source_registry.json'
registry = json.loads(registry_path.read_text(encoding='utf-8'))
sources = registry['source_families']

print(f"source families: {len(sources)}")
print(f"domains: {', '.join(registry['domain_labels'])}")

source families: 14
domains: hardware, os_kernel_drivers, application_software, network_connectivity, identity_access_accounts, storage_data_backup, security_malware, firmware_bios_uefi


In [3]:
import pandas as pd

pd.DataFrame([
    {
        'priority': s['priority'],
        'id': s['id'],
        'status': s['status'],
        'type': s['source_type'],
        'commercial_posture': s['commercial_posture'],
        'domains': ', '.join(s['target_domains']),
    }
    for s in sources
]).sort_values('priority')

,priority,id,status,type,commercial_posture,domains
0,1,stack_exchange_it_support_sites,candidate,qa,commercially_constrained_share_alike,"hardware, os_kernel_drivers, application_softw..."
7,1,public_it_helpdesk_ticket_datasets,candidate,ticket_dataset,license_varies_capture_per_dataset,"hardware, os_kernel_drivers, application_softw..."
12,1,cisa_kev_catalog,candidate,security_metadata,commercially_usable_public_domain_with_attribu...,"security_malware, firmware_bios_uefi, network_..."
11,1,nvd_security_feeds,candidate,security_metadata,commercially_usable_with_attribution_notice_an...,"security_malware, firmware_bios_uefi, applicat..."
13,1,github_advisory_database,candidate,security_metadata,commercially_usable_with_attribution,"security_malware, application_software"
8,2,microsoftdocs_github_repos,candidate,vendor_documentation_source_repo,repo_license_review_required,"os_kernel_drivers, application_software, netwo..."
1,2,ubuntu_official_and_community_docs,candidate,documentation,commercially_constrained_share_alike,"os_kernel_drivers, application_software, netwo..."
2,3,microsoft_learn_support_docs,candidate,vendor_documentation,poc_only_review_before_training,"os_kernel_drivers, application_software, netwo..."
10,3,archwiki_docs,candidate,documentation,poc_only_until_gfdl_handling_is_explicit,"hardware, os_kernel_drivers, application_softw..."
9,3,fedora_docs,candidate,documentation,commercially_constrained_share_alike,"os_kernel_drivers, application_software, netwo..."


## Acquisition Controls

Keep network work explicit. Start with one vertical slice, then scale only after raw capture, metadata, and normalization look right.

In [4]:
RUN_NETWORK_DOWNLOADS = False
SELECTED_SOURCE_ID = 'stack_exchange_it_support_sites'

selected = next(s for s in sources if s['id'] == SELECTED_SOURCE_ID)
selected

{'id': 'stack_exchange_it_support_sites',
 'name': 'Stack Exchange IT-support-adjacent Q&A',
 'status': 'candidate',
 'priority': 1,
 'source_type': 'qa',
 'acquisition_mode': 'data_dump_or_api',
 'seed_urls': ['https://stackoverflow.com/help/licensing',
  'https://askubuntu.com/help/licensing',
  'https://serverfault.com/',
  'https://superuser.com/',
  'https://unix.stackexchange.com/',
  'https://networkengineering.stackexchange.com/',
  'https://security.stackexchange.com/'],
 'license': 'CC BY-SA 2.5/3.0/4.0 depending on contribution date',
 'commercial_posture': 'commercially_constrained_share_alike',
 'attribution_required': True,
 'share_alike_required': True,
 'target_domains': ['hardware',
  'os_kernel_drivers',
  'application_software',
  'network_connectivity',
  'identity_access_accounts',
  'storage_data_backup',
  'security_malware',
  'firmware_bios_uefi'],
 'notes': 'High-value troubleshooting language and solved cases. Preserve post ids, author/display metadata where 

In [5]:
for path in [RAW_DIR, PROCESSED_DIR, ATTRIBUTION_DIR]:
    print(path.relative_to(PROJECT_ROOT), 'exists=', path.exists())

if not RUN_NETWORK_DOWNLOADS:
    print('Network acquisition disabled. Flip RUN_NETWORK_DOWNLOADS only for the selected source slice.')

data\raw exists= True
data\processed exists= True
data\attribution exists= True
Network acquisition disabled. Flip RUN_NETWORK_DOWNLOADS only for the selected source slice.


## First Vertical Slice

Recommended first slice: Stack Exchange IT-support-adjacent Q&A metadata only. Before downloading content, decide exact sites, dump/API route, fields to preserve, attribution output, and domain coverage targets.

## Active Slice Inspection

The active Stack Exchange slice has been inspected before normalization. Use these cells to review the evidence and keep downstream decisions tied to observed data.

In [6]:
ACTIVE_RUN_ID = '20260506T100325Z'
active_processed = PROCESSED_DIR / 'stack_exchange_it_support_sites' / ACTIVE_RUN_ID
inspection_summary = json.loads((active_processed / 'inspection_summary.json').read_text(encoding='utf-8'))
print(f"run: {inspection_summary['run_id']}")
print(f"questions: {inspection_summary['question_count']}")
print(f"answers: {inspection_summary['answer_count']}")
print(f"attribution rows: {inspection_summary['attribution_rows']}")
active_processed


run: 20260506T100325Z
questions: 27
answers: 276
attribution rows: 303


WindowsPath('c:/Users/nwagb/Desktop/it_support_ai/data/processed/stack_exchange_it_support_sites/20260506T100325Z')

In [7]:
pd.DataFrame(inspection_summary['query_results']).sort_values(['site', 'tag'])

,file,site,tag,domains,items,has_more,quota_remaining
0,questions_askubuntu_drivers.json,askubuntu,drivers,"[os_kernel_drivers, hardware]",3,True,266
1,questions_askubuntu_networking.json,askubuntu,networking,[network_connectivity],3,True,266
2,questions_security_malware.json,security,malware,[security_malware],3,True,260
3,questions_serverfault_active-directory.json,serverfault,active-directory,[identity_access_accounts],3,True,266
4,questions_serverfault_dns.json,serverfault,dns,[network_connectivity],0,False,266
5,questions_superuser_bios.json,superuser,bios,"[hardware, firmware_bios_uefi]",3,True,266
6,questions_superuser_hard-drive.json,superuser,hard-drive,"[hardware, storage_data_backup]",3,True,269
7,questions_superuser_networking.json,superuser,networking,[network_connectivity],3,True,269
8,questions_superuser_windows-10.json,superuser,windows-10,"[os_kernel_drivers, application_software]",3,True,269
9,questions_unix_permissions.json,unix,permissions,"[identity_access_accounts, os_kernel_drivers]",3,True,266


In [8]:
inspection_summary['text_quality'], inspection_summary['answer_strategy']

({'question_text_chars': {'min': 81,
   'p25': 235,
   'median': 418,
   'p75': 738,
   'max': 4421},
  'answer_text_chars': {'min': 32,
   'p25': 222,
   'median': 537,
   'p75': 1088,
   'max': 22671},
  'code_heavy_question_count': 6,
  'image_question_count': 1,
  'short_question_count': 1,
  'long_answer_count': 23},
 {'questions_with_accepted_answer_captured': 27,
  'questions_missing_accepted_answer_in_capture': 0,
  'accepted_missing_question_ids': [],
  'questions_where_accepted_is_top': 22,
  'questions_where_top_differs_from_accepted': 5,
  'captured_answers_per_question': {'min': 1, 'median': 9, 'max': 27}})

In [9]:
pd.DataFrame(inspection_summary['records'])[[
    'site', 'query_tag', 'question_id', 'answers_captured', 'accepted_captured',
    'accepted_is_top', 'question_text_chars', 'title'
]].sort_values(['site', 'query_tag', 'question_id'])

,site,query_tag,question_id,answers_captured,accepted_captured,accepted_is_top,question_text_chars,title
0,askubuntu,drivers,23238,7,True,True,172,How can I find what video driver is in use on ...
1,askubuntu,drivers,61396,14,True,True,235,How do I install the Nvidia drivers?
2,askubuntu,drivers,528293,16,True,False,738,"Is there a way to ""restart"" the touchpad driver?"
3,askubuntu,networking,95910,23,True,True,131,Command for determining my public IP?
4,askubuntu,networking,257263,27,True,True,135,How to display network traffic in the terminal?
5,askubuntu,networking,278448,9,True,True,505,How to know what program is listening on a giv...
7,security,malware,120748,10,True,True,1733,What should you do if you catch encryption ran...
6,security,malware,159331,10,True,True,449,"How is the ""WannaCry"" Malware spreading and ho..."
8,security,malware,199971,9,True,True,754,Search for military installed backdoors on laptop
11,serverfault,active-directory,49405,9,True,False,306,Command line to list users in a Windows Active...


Inspection artifacts:

- `data/processed/stack_exchange_it_support_sites/20260506T100325Z/inspection_report.md`
- `data/processed/stack_exchange_it_support_sites/20260506T100325Z/inspection_summary.json`
- `data/processed/stack_exchange_it_support_sites/20260506T100325Z/normalized_schema_v0.json`
- `data/processed/stack_exchange_it_support_sites/20260506T100325Z/tag_replacement_probe.json`

- `data/processed/stack_exchange_it_support_sites/20260506T100325Z/normalized_preview_v0.jsonl`

## Visible Acquisition Runs

Run acquisition in phases so progress is visible and interruption-safe. The scripts now show an overall artifact progress bar, per-file download bars for large files, and write live status under `data/processed/broad_data_wave/<run_id>/live_status.md`.

In [10]:
from datetime import datetime, timezone

PYTHON_EXE = PROJECT_ROOT / '.it_support' / 'Scripts' / 'python.exe'
INTERRUPTED_BROAD_RUN_ID = '20260506T145342Z'

phase_args = {
    'resume_docs_security': [
        '--include-large', '--skip-repo-zips', '--skip-hf', '--skip-zenodo'
    ],
    'repo_snapshots': [
        '--skip-doc-pages', '--skip-feeds', '--skip-hf', '--skip-zenodo'
    ],
    'ticket_datasets': [
        '--skip-doc-pages', '--skip-feeds', '--skip-repo-zips'
    ],
}

phase_expected = {
    'resume_docs_security': 103,
    'repo_snapshots': 8,
    'ticket_datasets': 8,  # Zenodo file downloads can increase this during the run.
}

def make_phase_command(phase, run_id=None):
    if run_id is None:
        run_id = INTERRUPTED_BROAD_RUN_ID if phase == 'resume_docs_security' else datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
    cmd = [
        str(PYTHON_EXE), 'scripts/acquire_broad_data_wave.py',
        '--run-id', run_id,
        '--no-progress',
        *phase_args[phase],
    ]
    return cmd, run_id

for name in phase_args:
    cmd, run_id = make_phase_command(name, INTERRUPTED_BROAD_RUN_ID if name == 'resume_docs_security' else '<new-run-id>')
    print(f"{name} [{phase_expected[name]} artifacts]: {' '.join(cmd)}")

resume_docs_security [103 artifacts]: c:\Users\nwagb\Desktop\it_support_ai\.it_support\Scripts\python.exe scripts/acquire_broad_data_wave.py --run-id 20260506T145342Z --no-progress --include-large --skip-repo-zips --skip-hf --skip-zenodo
repo_snapshots [8 artifacts]: c:\Users\nwagb\Desktop\it_support_ai\.it_support\Scripts\python.exe scripts/acquire_broad_data_wave.py --run-id <new-run-id> --no-progress --skip-doc-pages --skip-feeds --skip-hf --skip-zenodo
ticket_datasets [8 artifacts]: c:\Users\nwagb\Desktop\it_support_ai\.it_support\Scripts\python.exe scripts/acquire_broad_data_wave.py --run-id <new-run-id> --no-progress --skip-doc-pages --skip-feeds --skip-repo-zips


In [11]:
# Flip this only when you are ready to run a phase and watch notebook-native progress bars.
RUN_NETWORK_DOWNLOADS = True
SELECTED_PHASE = 'resume_docs_security'  # resume_docs_security, repo_snapshots, ticket_datasets

cmd, run_id = make_phase_command(SELECTED_PHASE)
live_dir = PROCESSED_DIR / 'broad_data_wave' / run_id
progress_path = live_dir / 'progress.jsonl'
current_path = live_dir / 'current_download.json'

def fmt_bytes(value):
    value = float(value or 0)
    for unit in ['B', 'KB', 'MB', 'GB']:
        if value < 1024 or unit == 'GB':
            return f'{value:.1f} {unit}'
        value /= 1024

def count_progress_rows(path):
    return sum(1 for _ in path.open(encoding='utf-8')) if path.exists() else 0

def read_json_if_exists(path):
    if not path.exists():
        return {}
    try:
        return json.loads(path.read_text(encoding='utf-8'))
    except json.JSONDecodeError:
        return {}

if RUN_NETWORK_DOWNLOADS:
    import html
    import os
    import subprocess
    import time
    import ipywidgets as widgets
    from IPython.display import display

    expected = phase_expected[SELECTED_PHASE]
    artifact_bar = widgets.IntProgress(value=0, min=0, max=expected, description='Artifacts')
    file_bar = widgets.IntProgress(value=0, min=0, max=100, description='Current')
    status = widgets.HTML(value=f'<b>Run:</b> {run_id}<br><b>Phase:</b> {SELECTED_PHASE}')
    file_status = widgets.HTML(value='Waiting for first download...')
    display(widgets.VBox([status, artifact_bar, file_bar, file_status]))

    env = os.environ.copy()
    env['PYTHONUNBUFFERED'] = '1'
    process = subprocess.Popen(cmd, cwd=PROJECT_ROOT, env=env, stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT)

    while process.poll() is None:
        done = count_progress_rows(progress_path)
        if done > artifact_bar.max:
            artifact_bar.max = done
        artifact_bar.value = done
        status.value = f'<b>Run:</b> {run_id}<br><b>Phase:</b> {SELECTED_PHASE}<br><b>Artifacts:</b> {done}/{artifact_bar.max}'

        current = read_json_if_exists(current_path)
        if current:
            total = int(current.get('total_bytes') or 0)
            done_bytes = int(current.get('bytes_done') or 0)
            if total > 0:
                file_bar.max = total
                file_bar.value = min(done_bytes, total)
            else:
                file_bar.max = 100
                file_bar.value = 0
            file_status.value = (
                f"<b>{html.escape(current.get('status', ''))}</b>: "
                f"{fmt_bytes(done_bytes)} / {fmt_bytes(total)}<br>"
                f"<code>{html.escape(current.get('path', ''))}</code>"
            )
        time.sleep(1)

    done = count_progress_rows(progress_path)
    artifact_bar.value = min(done, artifact_bar.max)
    artifact_bar.bar_style = 'success' if process.returncode == 0 else 'danger'
    status.value += f'<br><b>Process exited:</b> {process.returncode}'
else:
    print('Network acquisition is disabled.')
    print('Set RUN_NETWORK_DOWNLOADS=True and choose SELECTED_PHASE when ready.')
    print('Command:')
    print(' '.join(cmd))
    print(f'Live status path: {live_dir / "live_status.md"}')

In [12]:
status_roots = sorted((PROCESSED_DIR / 'broad_data_wave').glob('*/live_status.md')) if (PROCESSED_DIR / 'broad_data_wave').exists() else []
if not status_roots:
    print('No broad-wave live status files yet.')
else:
    latest_status = status_roots[-1]
    print(latest_status.relative_to(PROJECT_ROOT))
    print(latest_status.read_text(encoding='utf-8'))

data\processed\broad_data_wave\20260506T145342Z\live_status.md
# Live Broad Acquisition Status

Run id: `20260506T145342Z`
Artifacts processed: 103/103
Successful/existing: 95
Failed/gated: 8
Bytes present: 495,226,637

## Latest Artifact

- Source: `nvd_security_feeds`
- Kind: `feed`
- Status: `existing`
- URL: https://nvd.nist.gov/feeds/json/cpematch/2.0/nvdcpematch-2.0.tar.gz
- Path: `data\raw\nvd_security_feeds\20260506T145342Z\nvdcpematch-2.0.tar.gz`

